# Hexagonal Pixel Grid Converter

This notebook explores hexagonal sampling as an alternative to square pixels. It implements axial-coordinate math, visualizes pointy-top and flat-top grids, converts an image into hex cells, compares cell sizes, and exports sampled hex color data.

In [ ]:
import json
import math
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

## Hexagonal Grid Mathematics - explain axial and cube coordinates

Hex grids are commonly expressed in **axial** coordinates `(q, r)`, which are a compact view of the full **cube** system `(x, y, z)` constrained by `x + y + z = 0`. Conversions between pixel space and fractional axial coordinates let us sample or render a hex lattice, while cube-style rounding snaps a point to the nearest valid hex.

In [ ]:
def hex_round(q, r):
    x = q
    z = r
    y = -x - z
    rx, ry, rz = round(x), round(y), round(z)
    dx, dy, dz = abs(rx - x), abs(ry - y), abs(rz - z)
    if dx > dy and dx > dz:
        rx = -ry - rz
    elif dy > dz:
        ry = -rx - rz
    else:
        rz = -rx - ry
    return int(rx), int(rz)

def hex_to_pixel(q, r, size, orientation='pointy'):
    if orientation == 'pointy':
        x = size * math.sqrt(3) * (q + r / 2)
        y = size * 1.5 * r
    else:
        x = size * 1.5 * q
        y = size * math.sqrt(3) * (r + q / 2)
    return x, y

def pixel_to_hex(x, y, size, orientation='pointy'):
    if orientation == 'pointy':
        q = (math.sqrt(3) / 3 * x - y / 3) / size
        r = (2 / 3 * y) / size
    else:
        q = (2 / 3 * x) / size
        r = (-x / 3 + math.sqrt(3) / 3 * y) / size
    return hex_round(q, r)

print('Pointy-top axial -> pixel: x = size * sqrt(3) * (q + r/2), y = size * 3/2 * r')
print('Flat-top axial   -> pixel: x = size * 3/2 * q, y = size * sqrt(3) * (r + q/2)')
print('Nearest hex to pixel (50, 50) with size=10:', pixel_to_hex(50, 50, 10, orientation='pointy'))

## Visualizing the Hex Grid

Before sampling an image, it helps to look at the geometry directly. The outline grid below shows pointy-top hexes with their axial coordinates, making it clear how rows are staggered and how the coordinate system progresses.

In [ ]:
def hex_corners(cx, cy, size, orientation='pointy'):
    start = math.pi / 6 if orientation == 'pointy' else 0.0
    return [
        (cx + size * math.cos(start + i * math.pi / 3), cy + size * math.sin(start + i * math.pi / 3))
        for i in range(6)
    ]

fig, ax = plt.subplots(figsize=(10, 10))
size = 30
origin_x, origin_y = 60, 60
for r in range(10):
    for q in range(10):
        cx, cy = hex_to_pixel(q, r, size, orientation='pointy')
        cx += origin_x
        cy += origin_y
        poly = Polygon(hex_corners(cx, cy, size, orientation='pointy'), fill=False, edgecolor='black', linewidth=1)
        ax.add_patch(poly)
        ax.text(cx, cy, f'({q},{r})', ha='center', va='center', fontsize=7)
ax.set_xlim(0, 650)
ax.set_ylim(650, 0)
ax.set_aspect('equal')
ax.set_title('Pointy-top hex grid with axial coordinates')
ax.axis('off')
plt.show()

## Converting an Image to Hex Grid - pointy-top

To make a hex mosaic, we place hex centers over the image, sample the underlying source color at each center, and draw a filled polygon for every cell. The result preserves the image content while replacing square pixels with a more isotropic lattice.

In [ ]:
def make_hex_test_image(size=256):
    x = np.linspace(0, 1, size)
    y = np.linspace(0, 1, size)
    xx, yy = np.meshgrid(x, y)
    base = np.dstack([
        0.5 + 0.5 * np.sin(2 * np.pi * xx),
        yy,
        1.0 - xx * 0.7,
    ])
    image = Image.fromarray((np.clip(base, 0, 1) * 255).astype(np.uint8), mode='RGB')
    draw = ImageDraw.Draw(image, 'RGBA')
    draw.ellipse((30, 30, 120, 120), fill=(255, 180, 80, 180))
    draw.rectangle((150, 40, 230, 120), fill=(80, 255, 180, 180))
    draw.polygon([(70, 210), (128, 130), (220, 220)], fill=(255, 90, 160, 180))
    return np.asarray(image)

def generate_hex_cells(width, height, size, orientation='pointy'):
    origin_x, origin_y = size * 2.0, size * 2.0
    cells = []
    q_max = int(width / max(size, 1)) + 6
    r_max = int(height / max(size, 1)) + 6
    for r in range(-3, r_max):
        for q in range(-3, q_max):
            cx, cy = hex_to_pixel(q, r, size, orientation=orientation)
            cx += origin_x
            cy += origin_y
            if -size <= cx <= width + size and -size <= cy <= height + size:
                cells.append((q, r, cx, cy))
    return cells

def render_hex_image(arr, hex_size=8, orientation='pointy', show_borders=False, ax=None, return_cells=False):
    h, w = arr.shape[:2]
    cells = generate_hex_cells(w, h, hex_size, orientation=orientation)
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    patches = []
    colors = []
    for q, r, cx, cy in cells:
        sx = int(np.clip(round(cx), 0, w - 1))
        sy = int(np.clip(round(cy), 0, h - 1))
        patches.append(Polygon(hex_corners(cx, cy, hex_size, orientation=orientation), closed=True))
        colors.append(arr[sy, sx] / 255.0)
    collection = PatchCollection(
        patches,
        facecolor=colors,
        edgecolor=(0.15, 0.15, 0.15, 0.9) if show_borders else 'none',
        linewidth=0.4 if show_borders else 0.0,
    )
    ax.add_collection(collection)
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.set_aspect('equal')
    ax.axis('off')
    if return_cells:
        return ax, cells
    return ax

hex_test = make_hex_test_image(256)
fig, ax = plt.subplots(figsize=(6, 6))
render_hex_image(hex_test, hex_size=8, orientation='pointy', show_borders=False, ax=ax)
ax.set_title('Image converted to a pointy-top hex grid (size=8)')
plt.show()

## Flat-top vs Pointy-top Orientation

Orientation changes how the lattice aligns to image structure. Pointy-top hexes create vertical columns with staggered rows, while flat-top hexes emphasize horizontal runs and a different directional feel.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
render_hex_image(hex_test, hex_size=8, orientation='pointy', ax=axes[0])
axes[0].set_title('Pointy-top')
render_hex_image(hex_test, hex_size=8, orientation='flat', ax=axes[1])
axes[1].set_title('Flat-top')
plt.tight_layout()
plt.show()

## Effect of Hex Size

Smaller hexes preserve more detail because the sampling density is higher. Larger hexes reduce the number of cells and push the output toward a bold posterized mosaic with more obvious geometry.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, size in zip(axes.ravel(), [4, 8, 16, 32]):
    render_hex_image(hex_test, hex_size=size, orientation='pointy', ax=ax)
    ax.set_title(f'Hex size = {size}')
plt.tight_layout()
plt.show()

## Hex Grid with Borders

Borders make the cell structure explicit and can push the result toward a stained-glass or tiled aesthetic. Without borders, the lattice is still present but the image reads more like a soft resampling process.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
render_hex_image(hex_test, hex_size=10, orientation='pointy', show_borders=False, ax=axes[0])
axes[0].set_title('No borders')
render_hex_image(hex_test, hex_size=10, orientation='pointy', show_borders=True, ax=axes[1])
axes[1].set_title('With visible borders')
plt.tight_layout()
plt.show()

## Comparison: Hex vs Square Pixels

A square-pixel comparison helps isolate the effect of lattice shape. By matching the approximate number of cells, we can compare how hexes soften directional bias relative to a conventional blocky resample.

In [ ]:
_, cells_for_count = render_hex_image(hex_test, hex_size=10, orientation='pointy', ax=plt.subplots(figsize=(1, 1))[1], return_cells=True)
plt.close('all')
hex_count = len(cells_for_count)
side = max(1, int(round(math.sqrt(hex_count))))

square_small = Image.fromarray(hex_test).resize((side, side), Image.Resampling.BILINEAR)
square_big = square_small.resize((256, 256), Image.Resampling.NEAREST)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
render_hex_image(hex_test, hex_size=10, orientation='pointy', ax=axes[0])
axes[0].set_title(f'Hex grid (~{hex_count} cells)')
axes[1].imshow(square_big)
axes[1].set_title(f'Square grid ({side}x{side} ≈ {side * side} cells)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Exporting Hex Data

Once a hex lattice is defined, it is easy to export the sampled colors as structured data. That can feed downstream tools for procedural art, shader experiments, or custom renderers that need axial coordinate input.

In [ ]:
export_cells = []
export_size = 12
origin_x, origin_y = 18, 18
for r in range(8):
    for q in range(8):
        cx, cy = hex_to_pixel(q, r, export_size, orientation='pointy')
        cx += origin_x
        cy += origin_y
        sx = int(np.clip(round(cx), 0, hex_test.shape[1] - 1))
        sy = int(np.clip(round(cy), 0, hex_test.shape[0] - 1))
        export_cells.append(((q, r), hex_test[sy, sx].tolist()))

export_dict = {f'({q},{r})': [int(v) for v in color] for (q, r), color in export_cells}
first_ten = dict(list(export_dict.items())[:10])
print(json.dumps(first_ten, indent=2))